In [ ]:
%load norm_attention.py

In [ ]:
import time
import torch

def benchmark_model_cpu(run_model, make_input, Ts, iters=5, warmup=1):
    """
    model      : torch.nn.Module
    Ts         : iterable of T values
    make_input : function(T) -> input tensor(s) for the model
    iters      : timing iterations per T
    warmup     : warmup iterations per T

    returns    : list of average times (seconds), same order as Ts
    """
    times = []

    with torch.no_grad():
        for T in Ts:
            print("running", T)
            q, k, v, mask = make_input(T)
            # warmup
            for _ in range(warmup):
                _ = run_model(q, k, v, mask)

            t0 = time.perf_counter()
            for _ in range(iters):
                _ = run_model(q, k, v, mask)
            t1 = time.perf_counter()

            times.append((t1 - t0) / iters)
    return times


In [ ]:
from torch.nn.attention import SDPBackend, sdpa_kernel
import torch.nn.functional as F

Ts = [i for i in range(100, 10000, 100)]
device = torch.device("cuda")

def make_input(T):
    mask = generate_mask(T).to(device)
    B, H = 2, 3
    D, M = 64, 4
    q = torch.randn(B, H, T, D, device=device)
    k = torch.randn(B, H, T, D, device=device)
    v = torch.randn(B, H, T, M, device=device)
    return q, k, v, mask

def run_naive_attention(q, k, v, mask):
    naive_norm_attention(q, k, v)

def run_non_causal_attention(q, k, v, mask):
    non_causal_norm_attention(q, k, v)

def run_causal_attention(q, k, v, mask):
    causal_norm_attention(q, k, v)

def run_normal_attention(q, k, v, mask):
    with sdpa_kernel(backends=[SDPBackend.FLASH_ATTENTION]):
        F.scaled_dot_product_attention(q, k, v)


print("running times naive")
times_naive = benchmark_model_cpu(run_naive_attention, make_input, Ts)
print("running times non_causal")
times_non_causal = benchmark_model_cpu(run_non_causal_attention, make_input, Ts)
print("running times causal")
times_causal = benchmark_model_cpu(run_causal_attention, make_input, Ts)
print("running times flash attention")
times_flash = benchmark_model_cpu(run_normal_attention, make_input, Ts)

In [ ]:
import matplotlib.pyplot as plt


def plot_model_times(Ts, times_list, labels):
    """
    Ts         : list of T values (x-axis)
    times_list : list of lists, one per model
    labels     : list of model names (same order)

    Example:
    plot_model_times(Ts, [times_a, times_b, times_c],
                     ["Model A", "Model B", "Model C"])
    """
    plt.figure()

    for times, label in zip(times_list, labels):
        plt.plot(Ts, times, marker="o", label=label)

    plt.xlabel("T")
    plt.ylabel("Time (seconds)")
    plt.legend()
    plt.grid(True)
    plt.show()

plot_model_times(
    Ts,
    [times_naive, times_non_causal, times_causal, times_flash],
    ["naive", "non_causal", "causal", "flash"],
)
